# Month 2 • Week 5 — API Keys & Secrets Management
## Member 7 teaching notebook (beginner-friendly, screen-share ready)

**Goal from the HARBOUR README:** know how to store API keys safely and rotate them; Friday build: move keys to environment variables and add a secret-scanning check.

> Safety rule for this lesson: use fake/demo values only. Never paste a real API key into this notebook, chat, screenshots, Git, or logs.

## 1. Learning outcomes
By the end, teammates should be able to:
- explain what an API key and a secret are;
- identify unsafe places to store keys;
- read a key from an environment variable without printing it;
- use a local `.env` file safely and keep it out of Git;
- explain least privilege, rotation, revocation, and secret scanning;
- respond to a leaked-key incident.

## 2. What is an API key?
An **API key** is a credential (like a special password) that an application uses to prove it is allowed to call a service. Example: our inbox triage app may need a key to call an AI provider.

**Everyday analogy:** a hotel room key card opens a particular door. An API key is like a digital key that may unlock access to a service—and sometimes paid usage.

A **secret** is any sensitive value that should not be public: API keys, passwords, database credentials, access tokens, signing keys, and private certificates.

**Why protect them?** If someone gets your key, they may use your account, consume your quota, access data, or incur charges.

## 3. Unsafe vs safe places to store secrets

| Unsafe | Safer approach |
|---|---|
| Hard-coded inside Python source | Read from environment/config at runtime |
| Committed to Git/GitHub | Keep local `.env` ignored by Git; use a secret manager in deployed environments |
| Shared in screenshots, chat, or email | Share access through approved secret storage, not the value |
| Printed to terminal or logs | Log status/metadata only—never the secret itself |
| One key used forever | Rotate regularly and immediately after suspected exposure |

**Important:** environment variables reduce accidental source-code exposure; they are not a complete secret vault. On a shared or compromised machine, environment values may still be exposed.

## 4. Demonstration: a deliberately unsafe example (do not use)

The following pattern is unsafe because the credential is visible in the code and can be copied, committed, or captured in screenshots. The string below is fake.

In [ ]:
# UNSAFE EXAMPLE — fake key only; never put a real key in source code
API_KEY = "demo-key-123-not-real"

# A real key hard-coded here could be leaked through Git, screenshots, or sharing.

### Why this is dangerous
If this file is pushed to a public repository, anyone may copy the key. Deleting it in a later commit does not erase it from Git history or copies/clones. Treat a leaked key as compromised: revoke/rotate it.

## 5. Environment variables: the safer pattern
An **environment variable** is a value supplied to a program by its operating environment, rather than written into the code. Our code asks for the value by name.

We will check whether a variable exists, but deliberately will not display its contents.

In [ ]:
import os

# Read a variable named DEMO_API_KEY. No real key is expected for this lesson.
demo_key = os.getenv("DEMO_API_KEY")

if demo_key:
    print("A demo key is configured. Value hidden.")
else:
    print("No demo key found. That's expected unless you set one.")

### Set a temporary demo environment variable

**macOS/Linux (in a terminal):**
```bash
export DEMO_API_KEY='fake-demo-value-not-a-real-key'
python your_script.py
```

**Windows PowerShell:**
```powershell
$env:DEMO_API_KEY = 'fake-demo-value-not-a-real-key'
python your_script.py
```

This temporary variable usually lasts only for that terminal session. Do not use a real key in a classroom demo.

## 6. Local development: `.env` file
A `.env` file stores local configuration in simple `NAME=value` lines. Many Python projects use `python-dotenv` to load it.

Example `.env` contents (fake value):
```dotenv
DEMO_API_KEY=fake-demo-value-not-real
```

Example code (requires `python-dotenv` installed):
```python
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env into environment for local development
key = os.getenv('DEMO_API_KEY')
print('Configured:', bool(key))  # never print the value
```

A `.env` file is convenient for local work, but it must not be committed or shared.

## 7. Keep `.env` out of Git

Add these lines to a project's `.gitignore` file:
```gitignore
# Local secrets and environment configuration
.env
.env.*
!.env.example
```

The `.env.example` file is a safe template containing variable names only—no secret values. For example:
```dotenv
# Copy this file to .env and fill in locally. Do not commit .env.
DEMO_API_KEY=
```

If `.env` was already tracked, adding it to `.gitignore` alone does not untrack it. You must remove it from tracking, and if it contained a real secret, revoke/rotate that secret.

## 8. Exercise: inspect the ignore rules
This cell checks whether `.env` is listed in a simple `.gitignore` file in the current folder. It is a teaching check—not a complete Git security scanner.

In [ ]:
from pathlib import Path

gitignore = Path('.gitignore')
if gitignore.exists():
    rules = gitignore.read_text(errors='replace').splitlines()
    print('.env rule present:', any(line.strip() == '.env' for line in rules))
else:
    print('No .gitignore found in this notebook folder. Create one and add .env.')

## 9. Safe configuration pattern (no secret printing)

Use a settings helper that fails clearly if a required variable is missing. This example still uses a fake key and makes no network call.

In [ ]:
import os

def get_required_secret(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f'Missing required configuration: {name}')
    return value

# For classroom use, keep this unset or use a fake value.
try:
    key = get_required_secret('DEMO_API_KEY')
    print('Secret configuration is present. Value is hidden.')
except RuntimeError as exc:
    print(exc)

## 10. What must never go into logs?
Never log the secret value. Avoid logging full authorization headers, request objects that contain credentials, or error messages that echo credentials.

Good log example:
```text
provider=example request_id=abc123 status=success duration_ms=420
```

Bad log example (do not imitate):
```text
api_key=sk-...
```

If you need to debug, log whether configuration exists, the provider name, status, request ID, duration, and safe error category—not the key.

## 11. Least privilege
**Least privilege** means give a credential only the permissions it needs, for only as long as needed.

Examples:
- If a service only needs to read, don't give it write permission.
- Use a key dedicated to the learning app instead of a personal, all-purpose key.
- Restrict by project, API, environment, IP, or quota when the provider supports it.
- Separate development, test, and production credentials.

**Trade-off:** restrictions reduce damage if a key leaks, but managing separate keys and permissions takes extra setup.

## 12. Rotation, revocation, and expiration
- **Rotation:** replace an old key with a new key on a planned schedule or after risk.
- **Revocation:** disable/delete a key so it can no longer be used.
- **Expiration:** a key becomes invalid automatically after a set date/time.

Typical safe rotation steps:
1. Create a new key with the minimum permissions.
2. Store it in the approved secret store/environment.
3. Deploy/configure the app to use the new key.
4. Verify the app works.
5. Revoke the old key.
6. Check logs/usage for unexpected activity.

If a key is exposed, do **not** wait for the normal rotation date. Revoke it promptly, create a replacement, inspect for misuse, and remove the leaked value from current files and history where appropriate.

## 13. Secret scanning
Secret scanning looks for patterns that may be credentials in files or Git changes. It can catch accidental commits, but it is not perfect: false positives happen and some secrets may not match known patterns.

For our Week 5 build, the README milestone is: **move keys to environment variables and add a secret-scanning check**. The team should choose and configure a scanner in the repository/CI, review its findings, and verify it blocks or flags a deliberate test secret without using any real credential.

Do not assume a clean scan proves there are no secrets.

## 14. Mini demo: a safe secret-check helper
This simple function is not a real scanner. It demonstrates the policy check we want: prevent obviously unsafe secret-like values from being placed in source. Real scanners use broader rules and repository history.

In [ ]:
def check_for_placeholder_secret(text: str) -> bool:
    # Teaching-only check: look for a fake hard-coded assignment.
    return 'API_KEY = "demo-key' in text or 'API_KEY=demo-key' in text

unsafe_sample = 'API_KEY = "demo-key-123-not-real"'
safe_sample = 'api_key = os.getenv("DEMO_API_KEY")'
print('Unsafe sample flagged:', check_for_placeholder_secret(unsafe_sample))
print('Safe environment lookup flagged:', check_for_placeholder_secret(safe_sample))

## 15. Practical scenario: our Inbox Triage Assistant

The assistant calls a hosted model provider. Its key is needed to authenticate requests.

**A safer flow:**
1. Developer creates a project-specific provider key.
2. Key is stored in local `.env` for development or a deployment secret manager for production.
3. Python reads the key from the environment.
4. The request uses the key in the provider SDK/HTTP authentication layer.
5. Logs capture safe request metadata, never the key.
6. CI scans the repository for leaked secrets.
7. The key is rotated/revoked when necessary.

**Analogy:** the application is a delivery driver; the key is the access card to the warehouse. The driver should carry only the access card needed for that route, not the master card.

## 16. Team exercise: spot the risk
For each situation, name the risk and the safer fix.

1. A developer pastes their API key into a public GitHub issue.
2. The app prints the full request headers when an API call fails.
3. A `.env` file is committed.
4. Everyone in the team shares one unrestricted key.
5. A key is leaked, but the developer only deletes the line and keeps using the same key.
6. The team stores an empty `.env.example` template in Git.

**Suggested answers:** 1) exposed credential—revoke/rotate and investigate; 2) logs may leak auth—redact headers; 3) secret in source control—remove tracking and rotate; 4) excessive access/shared accountability—use separate restricted keys; 5) deletion doesn't invalidate key—revoke/rotate and inspect history/usage; 6) usually okay if it contains names/placeholders only.

## 17. Friday mini-project build checklist (from the weekly goal)

- [ ] Find where the app currently reads the provider key.
- [ ] Remove any hard-coded credential (only if one exists; never copy it into the notebook).
- [ ] Read the key from an environment variable.
- [ ] Create/update `.env.example` with names only.
- [ ] Add `.env` and related local secret files to `.gitignore`.
- [ ] Confirm the actual `.env` is not tracked by Git.
- [ ] Add a secret-scanning check to the repository/CI.
- [ ] Test the scan with a scanner's safe test fixture or dummy pattern.
- [ ] Review logs to confirm no secret value appears.
- [ ] Explain the steps to rotate a key.

**Definition of done:** code reads secrets from environment; secrets aren't committed; a secret scan is present; the team can demonstrate and explain how it would respond to a leak.

## 18. Questions to ask the team (presentation practice)
1. What is an API key?
2. Why is it similar to a password?
3. What can happen if an API key leaks?
4. What is a secret? Give three examples.
5. What is an environment variable?
6. Why is hard-coding a key risky?
7. What is `.env` used for?
8. Why should `.env` be in `.gitignore`?
9. What belongs in `.env.example`?
10. If `.env` was committed, is adding `.gitignore` enough? Why not?
11. Why must we avoid printing keys in logs?
12. Explain least privilege.
13. What is rotation?
14. What is revocation?
15. What should happen immediately after a key is leaked?
16. What does secret scanning do?
17. Can secret scanning catch every leak?
18. Why separate dev/test/prod keys?
19. Why is an environment variable not a complete secret-management solution?
20. What is the Friday deliverable for our team?

### Short answer guide
- API key: credential used to authenticate/authorize app access.
- Leak impact: unauthorized use, data access, quota/charges.
- Secret: sensitive value that must be protected.
- Environment variable: runtime configuration provided outside source code.
- `.env`: local configuration file; do not commit.
- `.env.example`: safe variable-name template.
- Least privilege: only necessary permissions.
- Rotation: replace key; revocation: disable old key.
- Leak response: revoke, replace, investigate, clean up, and monitor.
- Scanner: finds suspicious credential patterns; imperfect.
- Week goal: move keys to env vars and add secret scanning.

## 19. 2-minute closing summary (say this aloud)

> An API key is like a password that lets our application access another service. If we expose it, someone else may use our access or spend our quota. We should not hard-code keys or commit them to Git. During local development, we can keep them in a `.env` file that Git ignores, and our code reads them through environment variables. We should give keys only the permissions they need, avoid logging their values, scan repositories for accidental leaks, and rotate or revoke a key immediately if it is exposed. Environment variables help, but production systems should use a dedicated secret-management solution where possible.

## 20. Presenter reminder
- Explain with the hotel key-card analogy first.
- Run only the safe demo cells.
- Never reveal a real key while sharing your screen.
- Ask the class to predict before running the exercise cells.
- Invite one teammate to explain the rotation steps back to you.
- Be honest: this notebook teaches the concepts and safe patterns; it does not make real provider calls or perform an actual secret scan.